# Group 5

## Topic: Object Detection
## Judul: Smart Traffic Monitoring dengan YOLOv8!

- Junico -  2802518383
- Anang Tantowi - 2802497492
- Dextra Faren - 2802493645
- Samuel Christopher - 2802545466
- Valentino Kurniawan - 2802524120

---

In [17]:
import os
import glob
import random
import yaml
import numpy as np
import pandas as pd

RANDOM_STATE = 42

# Set environment variables before importing torch so CUDA picks them up correctly.
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch
import torchvision
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# Reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Folder untuk menyimpan semua PNG hasil training & comparison
EXPORT_DIR = 'exported_plots'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Helper agar setiap plot disimpan dengan kualitas konsisten (DPI 300 = print-ready)
def save_fig(fig, filename, dpi=300):
    """Save figure ke folder EXPORT_DIR dengan DPI tinggi."""
    path = os.path.join(EXPORT_DIR, filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    print(f"  -> saved: {path}")
    return path

print(f"Seed:        {RANDOM_STATE}")
print(f"Torch:       {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA:        {torch.cuda.is_available()} (built with {torch.version.cuda})")
print(f"Device used: {DEVICE}")
print(f"Export dir:  {EXPORT_DIR}/")


Seed:        42
Torch:       2.5.1+cu121
Torchvision: 0.20.1+cu121
CUDA:        True (built with 12.1)
Device used: cuda
Export dir:  exported_plots/


In [ ]:
DATA_PATH = r"C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL/data.yaml"

# CATATAN: blok ini OVERWRITE data.yaml setiap kali dijalankan.
# Pastikan ya, path train/val/test memang relatif terhadap lokasi data.yaml.
with open(DATA_PATH, 'r') as f:
    data_config = yaml.safe_load(f)

data_config['train'] = 'train/images'
data_config['val']   = 'valid/images'
data_config['test']  = 'test/images'

with open(DATA_PATH, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("data.yaml updated!")
print(data_config)


data.yaml updated!
{'names': ['Angkot', 'Becak', 'Bus Besar', 'Bus Sedang', 'Gerobag', 'Gerobak', 'Mobil', 'Motor', 'Pick Up', 'Plat', 'Sepeda', 'Truk Besar', 'Truk Sedang'], 'nc': 13, 'roboflow': {'license': 'CC BY 4.0', 'project': 'deteksi-kendaraan', 'url': 'https://universe.roboflow.com/irfan-maulana/deteksi-kendaraan/dataset/5', 'version': 5, 'workspace': 'irfan-maulana'}, 'test': 'test/images', 'train': 'train/images', 'val': 'valid/images'}


In [ ]:
# 1. PREPROCESSING & DATA SPLITTING 
print("=== Dataset Configuration ===")
print(f"Classes ({data_config['nc']}): {data_config['names']}")

split_counts = {}
for split in ['train', 'val', 'test']:
    img_dir = data_config[split]
    if not os.path.isabs(img_dir):
        img_dir = os.path.join(os.path.dirname(DATA_PATH), img_dir)
    n = len(glob.glob(os.path.join(img_dir, '*.*')))
    split_counts[split] = n
    print(f"  {split:5s}: {n} images")

print("\n=== Preprocessing ===")
print("- Source: Roboflow")
print("- Format: YOLO (.txt labels)")
print("- Resize: 640x640 (handled by Ultralytics)")
print("- Augmentations: default Ultralytics pipeline")


=== Dataset Configuration ===
Classes (13): ['Angkot', 'Becak', 'Bus Besar', 'Bus Sedang', 'Gerobag', 'Gerobak', 'Mobil', 'Motor', 'Pick Up', 'Plat', 'Sepeda', 'Truk Besar', 'Truk Sedang']
  train: 1347 images
  val  : 440 images
  test : 212 images

=== Preprocessing ===
- Source: Roboflow
- Format: YOLO (.txt labels)
- Resize: 640x640 (handled by Ultralytics)
- Augmentations: default Ultralytics pipeline


# Preprocessing and Dataset Insights

## Dataset Configuration

The dataset uses YOLO object detection format with image files and `.txt` annotation labels. The class names and the number of train, validation, and test images are displayed dynamically in the code cell above, so the explanation remains valid even if the dataset size changes.

The data is divided into **train / validation / test** splits. The training set is used to teach the model object patterns, the validation set monitors learning during training, and the test set is used for final evaluation on unseen data.

Although the dataset is still considered relatively limited for deep learning object detection, transfer learning from pretrained YOLO weights helps the model learn faster. Because of this, preprocessing, augmentation, and model-size selection are important to reduce overfitting and improve generalization.

---

## Preprocessing Analysis

The dataset was obtained in YOLO format, which is directly compatible with the Ultralytics YOLO training pipeline. This makes the training process simpler because bounding box annotations are already stored in the expected format.

During training, all images are resized to **640×640 pixels**. This standardization helps the model process images consistently and improves batch training efficiency. YOLO also applies built-in augmentation such as mosaic, flip, and HSV augmentation during training. These augmentations are useful because they simulate different lighting, positions, and object variations.

---

## Relevance to the Case

For vehicle and license plate detection, preprocessing is important because the objects can appear at different sizes and angles. License plates are usually smaller than vehicles, so consistent resizing and augmentation help the model learn more robust visual features.


In [ ]:
# 2. MAIN MODEL: YOLOv8s 
model = YOLO('yolov8s.pt')  # pretrained on COCO

print("=== Model Architecture ===")
print("- Model: YOLOv8s (Small)")
print("- Pretrained: COCO")
print("- Task: Object Detection")
print(f"- Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

=== Model Architecture ===
- Model: YOLOv8s (Small)
- Pretrained: COCO
- Task: Object Detection
- Parameters: 11,166,560


In [6]:
results = model.train(
    data=DATA_PATH,
    epochs=30,
    imgsz=640,
    batch=8,
    name='vehicle_detection',
    patience=10,
    device=DEVICE,
    workers=0,
    seed=RANDOM_STATE,
    deterministic=True,
    exist_ok=True,
)

RESULTS_DIR = str(results.save_dir)
print(f"\nYOLOv8s training output dir: {RESULTS_DIR}")


New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.50  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, mom

c:\Users\Asus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Overriding model.yaml nc=80 with nc=13

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytic

c:\Users\Asus\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/06/05 22:27:40 INFO mlflow.tracking.fluent: Experiment with name '/Shared/Ultralytics' does not exist. Creating a new experiment.
2026/06/05 22:27:40 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<

MLflow: logging run_id(a1415e2a410e41c8aa8d30fb7112e346) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL\runs\detect\vehicle_detection
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/30      1.94G      1.404      2.078      1.264         24        640: 100% ━━━━━━━━━━━━ 169/169 2.4it/s 1:090.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 28/28 3.2it/s 8.9s0.3s
                   all        440       2925       0.51      0.407      0.419      0.285

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      1.96G      1.309      1.378      1.196         16        640: 10

# Model and Training Insights

## Model Configuration

The main model used in this project is **YOLOv8s**, pretrained on the COCO dataset. A lighter **YOLOv8n** model is also added later as a comparison model for bonus-point analysis. YOLOv8s was selected because it provides a good balance between detection accuracy and computational efficiency. With approximately **11.1 million parameters**, the model is lightweight enough for faster training and inference while still being capable of learning complex object detection features.

Using pretrained weights from COCO helps the model learn faster because it already understands general visual patterns such as vehicles, people, and object shapes. This transfer learning approach is especially useful since the dataset size is relatively limited.

---

## Training Configuration

The model was trained for **30 epochs** with:
- image size: **640×640**
- batch size: **8**
- early stopping patience: **10**
- device: **CUDA (GPU)**
- seed: **42**
- deterministic mode: **enabled**

The batch size of 8 was chosen to balance GPU memory usage and training stability. Meanwhile early stopping helps prevent unnecessary training once the validation performance stops improving.

---



In [ ]:
#  3. TRAINING & VALIDATION RESULTS - YOLOv8s 
# Ultralytics otomatis menyimpan results.png (kurva loss & metrics per epoch)
# di folder RESULTS_DIR. Di sini kita load dan re-export dengan title + DPI tinggi.

results_img_path = os.path.join(RESULTS_DIR, 'results.png')
results_img = Image.open(results_img_path)

fig, ax = plt.subplots(figsize=(15, 8))
ax.imshow(results_img)
ax.axis('off')
ax.set_title('YOLOv8s - Training & Validation Curves', fontsize=14, fontweight='bold')
save_fig(fig, 'yolov8s_training_results.png')
plt.show()


  -> saved: exported_plots\yolov8s_training_results.png


<Figure size 1500x800 with 1 Axes>

# Training and Validation Insights

The training and validation results show that the YOLOv8s model learned consistently throughout the training process. Most training losses, including `train/box_loss`, `train/cls_loss`, and `train/dfl_loss`, gradually decreased over epochs. This indicates that the model became better at localizing objects, classifying object categories, and refining bounding box predictions.

The validation losses also showed a similar downward trend without significant divergence from the training losses. This is an important indication that the model did not experience severe overfitting. The validation curves remained relatively stable, meaning the model was able to generalize reasonably well to unseen data.

The precision metric fluctuated during training but overall showed an increasing trend, reaching around **0.83** near the final epochs. This means the model became more accurate in predicting objects with fewer false positives. Recall also improved gradually to around **0.76**, indicating that the model detected more target objects over time.

For detection performance, both `mAP50` and `mAP50-95` consistently increased during training. The final **mAP50 reached 0.831**, while **mAP50-95 reached 0.577**. The gap between these two values is expected: mAP50 only requires loose bounding box overlap (IoU 0.5), while mAP50-95 averages across stricter thresholds up to 0.95, so it is always lower. This suggests the model detects objects well but still has room to improve on precise localization.

Some fluctuations in the metric curves are still visible, particularly in precision and recall. This is normal for smaller datasets because the model becomes more sensitive to variations between batches and object distributions.

Overall, the training results indicate that the model converged properly and achieved stable learning performance without major signs of instability or overfitting.

In [ ]:
# 4. EVALUATION ON TEST SET - YOLOv8s 
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, 'weights', 'best.pt')
best_model = YOLO(BEST_MODEL_PATH)

metrics = best_model.val(
    data=DATA_PATH,
    split='test',
    seed=RANDOM_STATE,
    deterministic=True,
    workers=0,
    name='vehicle_detection_yolov8s_test_eval',
    exist_ok=True,
)
TEST_EVAL_DIR = str(metrics.save_dir)

print("=== Test Set Evaluation: YOLOv8s ===")
print(f"  mAP50:     {metrics.box.map50:.4f}")
print(f"  mAP50-95:  {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall:    {metrics.box.mr:.4f}")
print(f"  Best weights: {BEST_MODEL_PATH}")

# Confusion matrix: prefer dari test-eval folder, fallback ke training folder
cm_path = os.path.join(TEST_EVAL_DIR, 'confusion_matrix.png')
if not os.path.exists(cm_path):
    cm_path = os.path.join(RESULTS_DIR, 'confusion_matrix.png')

if os.path.exists(cm_path):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(Image.open(cm_path))
    ax.axis('off')
    ax.set_title('Confusion Matrix - YOLOv8s', fontsize=14, fontweight='bold')
    save_fig(fig, 'yolov8s_confusion_matrix.png')
    plt.show()
else:
    print("  WARNING: confusion_matrix.png tidak ditemukan.")


Ultralytics 8.4.50  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
Model summary (fused): 73 layers, 11,130,615 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 3.10.8 MB/s, size: 51.9 KB)
val: Scanning C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL\test\labels... 212 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 212/212 235.3it/s 0.9s0.0s
val: New cache created: C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 2.5it/s 5.6s0.4s
                   all        212       1432      0.827      0.756      0.831      0.577
                Angkot          8          8          1       0.37      0.654      0.569
                 Becak         19         22      0.743      0.818      0.828      0.553
             Bus Besar          8          9      0.875      0.776     

<Figure size 800x800 with 1 Axes>

## Training Performance Analysis

Use the printed metrics from the evaluation cell above as the final performance values for the main model. The most important metrics for presentation are:

- **mAP50**: detection accuracy with IoU threshold 0.50
- **mAP50-95**: stricter detection accuracy across multiple IoU thresholds
- **Precision**: how many predicted objects are correct
- **Recall**: how many real objects are successfully detected

A good conclusion should not only mention the highest metric, but also explain the trade-off. For example, a model with higher mAP is usually better for accuracy, while a smaller model can be better for faster deployment.

For this project, special attention should be given to the `Plat` (license plate) class because license plates are smaller objects and are usually harder to localize precisely compared with larger vehicle classes such as `Mobil`, `Bus Besar`, or `Truk Besar`. Minority classes with few samples, such as `Becak`, `Gerobak`, and `Gerobag`, should also be noted, since limited data makes their precision and recall less reliable.

# Bonus Model Comparison: YOLOv8s vs YOLOv8n

To support the presentation criteria and potentially gain bonus points, this notebook compares the main model (**YOLOv8s**) with a lighter baseline model (**YOLOv8n**).

**Why YOLOv8n?**

YOLOv8n is smaller and faster than YOLOv8s, so it is suitable as a comparison model for a dataset of around 2,000 images and for deployment on limited hardware. The comparison helps show whether the extra complexity of YOLOv8s is worth it compared with a lighter model.

This section uses the same dataset, image size, epochs, batch size, seed, and deterministic setting to make the comparison fair.


In [ ]:
# 5. MODEL COMPARISON: YOLOv8n TRAINING + EVAL 
# Dilatih dengan hyperparameter IDENTIK ke YOLOv8s supaya perbandingan fair.

EPOCHS, IMG_SIZE, BATCH_SIZE, PATIENCE = 30, 640, 8, 10

print("=== Training YOLOv8n (comparison model) ===")
yolov8n_model = YOLO('yolov8n.pt')

nano_results = yolov8n_model.train(
    data=DATA_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name='vehicle_detection_yolov8n',
    patience=PATIENCE,
    device=DEVICE,
    workers=0,
    seed=RANDOM_STATE,
    deterministic=True,
    exist_ok=True,
)
NANO_RESULTS_DIR = str(nano_results.save_dir)
NANO_BEST_MODEL_PATH = os.path.join(NANO_RESULTS_DIR, 'weights', 'best.pt')

# Evaluate di test split yang sama
nano_best_model = YOLO(NANO_BEST_MODEL_PATH)
nano_metrics = nano_best_model.val(
    data=DATA_PATH,
    split='test',
    seed=RANDOM_STATE,
    deterministic=True,
    workers=0,
    name='vehicle_detection_yolov8n_test_eval',
    exist_ok=True,
)
NANO_TEST_EVAL_DIR = str(nano_metrics.save_dir)

print("\n=== Test Set Evaluation: YOLOv8n ===")
print(f"  mAP50:     {nano_metrics.box.map50:.4f}")
print(f"  mAP50-95:  {nano_metrics.box.map:.4f}")
print(f"  Precision: {nano_metrics.box.mp:.4f}")
print(f"  Recall:    {nano_metrics.box.mr:.4f}")


=== Training YOLOv8n (comparison model) ===
New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.50  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, 

In [ ]:
# 5b. TRAINING & VALIDATION RESULTS - YOLOv8n 
# Visualisasi yang sama seperti YOLOv8s, supaya kedua model bisa dibandingkan visual.

nano_results_img_path = os.path.join(NANO_RESULTS_DIR, 'results.png')
fig, ax = plt.subplots(figsize=(15, 8))
ax.imshow(Image.open(nano_results_img_path))
ax.axis('off')
ax.set_title('YOLOv8n - Training & Validation Curves', fontsize=14, fontweight='bold')
save_fig(fig, 'yolov8n_training_results.png')
plt.show()

# Confusion matrix YOLOv8n
nano_cm_path = os.path.join(NANO_TEST_EVAL_DIR, 'confusion_matrix.png')
if not os.path.exists(nano_cm_path):
    nano_cm_path = os.path.join(NANO_RESULTS_DIR, 'confusion_matrix.png')

if os.path.exists(nano_cm_path):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(Image.open(nano_cm_path))
    ax.axis('off')
    ax.set_title('Confusion Matrix - YOLOv8n', fontsize=14, fontweight='bold')
    save_fig(fig, 'yolov8n_confusion_matrix.png')
    plt.show()


  -> saved: exported_plots\yolov8n_training_results.png


<Figure size 1500x800 with 1 Axes>

  -> saved: exported_plots\yolov8n_confusion_matrix.png


<Figure size 800x800 with 1 Axes>

In [ ]:
# 5c. SIDE-BY-SIDE TRAINING CURVES (1 PNG) 
# Memudahkan pembaca poster melihat YOLOv8s vs YOLOv8n dalam 1 figure.

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
axes[0].imshow(Image.open(results_img_path))
axes[0].axis('off')
axes[0].set_title('YOLOv8s - Training Curves', fontsize=13, fontweight='bold')

axes[1].imshow(Image.open(nano_results_img_path))
axes[1].axis('off')
axes[1].set_title('YOLOv8n - Training Curves', fontsize=13, fontweight='bold')

plt.suptitle('Side-by-Side Training Comparison', fontsize=15, fontweight='bold', y=1.02)
save_fig(fig, 'training_curves_side_by_side.png')
plt.show()


  -> saved: exported_plots\training_curves_side_by_side.png


<Figure size 2000x700 with 2 Axes>

In [ ]:
# 6. MODEL COMPARISON TABLE 

def count_parameters(yolo_model):
    return sum(p.numel() for p in yolo_model.model.parameters())

def safe_metric(value):
    try:
        return float(value)
    except Exception:
        return np.nan

def build_row(model_name, role, model_path, metrics_obj, notes):
    loaded = YOLO(model_path)
    speed = getattr(metrics_obj, 'speed', {}) or {}
    return {
        'Model': model_name,
        'Role': role,
        'Parameters': count_parameters(loaded),
        'mAP50': round(safe_metric(metrics_obj.box.map50), 4),
        'mAP50-95': round(safe_metric(metrics_obj.box.map), 4),
        'Precision': round(safe_metric(metrics_obj.box.mp), 4),
        'Recall': round(safe_metric(metrics_obj.box.mr), 4),
        'Inference (ms/img)': round(safe_metric(speed.get('inference', np.nan)), 3),
        'Best Weights Path': model_path,
        'Notes': notes,
    }

comparison_rows = [
    build_row('YOLOv8s', 'Main model', BEST_MODEL_PATH, metrics,
              'Higher capacity, candidate utama jika akurasi prioritas.'),
    build_row('YOLOv8n', 'Comparison model', NANO_BEST_MODEL_PATH, nano_metrics,
              'Lebih ringan & cepat, cocok untuk deployment terbatas.'),
]

model_comparison = pd.DataFrame(comparison_rows)
model_comparison['Rank by mAP50-95'] = (
    model_comparison['mAP50-95'].rank(ascending=False, method='min').astype(int)
)
model_comparison = (
    model_comparison.sort_values(by=['mAP50-95', 'mAP50'], ascending=False)
                    .reset_index(drop=True)
)

print("=== Model Comparison Table ===")
display(model_comparison)

model_comparison.to_csv(os.path.join(EXPORT_DIR, 'model_comparison_results.csv'), index=False)
print(f"  -> saved: {EXPORT_DIR}/model_comparison_results.csv")


=== Model Comparison Table ===


,Model,Role,Parameters,mAP50,mAP50-95,Precision,Recall,Inference (ms/img),Best Weights Path,Notes,Rank by mAP50-95
0,YOLOv8s,Main model,11140631,0.8305,0.5766,0.8270,0.7564,17.729,C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)...,"Higher capacity, candidate utama jika akurasi ...",1
1,YOLOv8n,Comparison model,3013383,0.7507,0.5230,0.7093,0.6957,8.210,C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)...,"Lebih ringan & cepat, cocok untuk deployment t...",2


  -> saved: exported_plots/model_comparison_results.csv


In [ ]:
# 6b. COMPARISON CHART (4-in-1) 
# 4 subplot: mAP50-95, mAP50, Inference speed, Parameter count.
# Dengan value label di atas tiap batang supaya angka langsung kebaca di poster.

def annotate_bars(ax, fmt='{:.3f}'):
    """Tulis nilai di atas tiap bar."""
    for bar in ax.patches:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h, fmt.format(h),
                ha='center', va='bottom', fontsize=10, fontweight='bold')

models = model_comparison['Model'].tolist()
colors = ['#2E86AB', '#E63946']  # biru = peringkat 1, merah = peringkat 2

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# (a) mAP50-95
axes[0, 0].bar(models, model_comparison['mAP50-95'], color=colors)
axes[0, 0].set_title('Accuracy: mAP50-95 (higher = better)', fontweight='bold')
axes[0, 0].set_ylabel('mAP50-95')
axes[0, 0].set_ylim(0, max(model_comparison['mAP50-95']) * 1.2)
axes[0, 0].grid(axis='y', alpha=0.3)
annotate_bars(axes[0, 0])

# (b) mAP50
axes[0, 1].bar(models, model_comparison['mAP50'], color=colors)
axes[0, 1].set_title('Accuracy: mAP50 (higher = better)', fontweight='bold')
axes[0, 1].set_ylabel('mAP50')
axes[0, 1].set_ylim(0, max(model_comparison['mAP50']) * 1.2)
axes[0, 1].grid(axis='y', alpha=0.3)
annotate_bars(axes[0, 1])

# (c) Inference speed (lower = better)
axes[1, 0].bar(models, model_comparison['Inference (ms/img)'], color=colors)
axes[1, 0].set_title('Speed: Inference Time (lower = better)', fontweight='bold')
axes[1, 0].set_ylabel('ms / image')
axes[1, 0].grid(axis='y', alpha=0.3)
annotate_bars(axes[1, 0], fmt='{:.2f}')

# (d) Parameter count
params_in_millions = model_comparison['Parameters'] / 1e6
axes[1, 1].bar(models, params_in_millions, color=colors)
axes[1, 1].set_title('Model Size: Parameters (lower = lighter)', fontweight='bold')
axes[1, 1].set_ylabel('Parameters (millions)')
axes[1, 1].grid(axis='y', alpha=0.3)
annotate_bars(axes[1, 1], fmt='{:.2f}M')

plt.suptitle('YOLOv8s vs YOLOv8n - Full Comparison', fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
save_fig(fig, 'model_comparison_4in1.png')
plt.show()

# Individual chart juga di-save (backward compatible dengan kode lama)
for col, fname, ylabel in [
    ('mAP50-95', 'model_comparison_map50_95.png', 'mAP50-95'),
    ('Inference (ms/img)', 'model_comparison_inference_speed.png', 'Inference Time (ms/img)'),
]:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(model_comparison['Model'], model_comparison[col], color=colors)
    ax.set_title(f'Model Comparison by {ylabel}')
    ax.set_xlabel('Model'); ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.3)
    annotate_bars(ax)
    plt.tight_layout()
    save_fig(fig, fname)
    plt.close(fig)


  -> saved: exported_plots\model_comparison_4in1.png


<Figure size 1300x1000 with 4 Axes>

  -> saved: exported_plots\model_comparison_map50_95.png
  -> saved: exported_plots\model_comparison_inference_speed.png


In [ ]:
# 6c. CONCLUSION & DEPLOY MODEL 
best_row = model_comparison.iloc[0]
DEPLOY_MODEL_NAME = best_row['Model']
DEPLOY_MODEL_PATH = best_row['Best Weights Path']

summary = f"""
=== Model Comparison Conclusion ===
Best model by mAP50-95: {DEPLOY_MODEL_NAME}
Best weights path:      {DEPLOY_MODEL_PATH}

Interpretation:
- YOLOv8s = lebih akurat, parameter lebih banyak.
- YOLOv8n = lebih ringan & cepat inference, baseline untuk deployment edge/low-resource.
- Untuk final presentation: pakai model dengan mAP50-95 tertinggi,
  lalu jelaskan trade-off menggunakan precision, recall, inference speed, dan param count.
"""
print(summary)

with open(os.path.join(EXPORT_DIR, 'model_comparison_summary.txt'), 'w', encoding='utf-8') as f:
    f.write(summary)
print(f"  -> saved: {EXPORT_DIR}/model_comparison_summary.txt")

# List semua file PNG yang sudah di-export
print("\n=== All exported PNG files ===")
for p in sorted(glob.glob(os.path.join(EXPORT_DIR, '*.png'))):
    print(f"  {p}")



=== Model Comparison Conclusion ===
Best model by mAP50-95: YOLOv8s
Best weights path:      C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL\runs\detect\vehicle_detection\weights\best.pt

Interpretation:
- YOLOv8s = lebih akurat, parameter lebih banyak.
- YOLOv8n = lebih ringan & cepat inference, baseline untuk deployment edge/low-resource.
- Untuk final presentation: pakai model dengan mAP50-95 tertinggi,
  lalu jelaskan trade-off menggunakan precision, recall, inference speed, dan param count.

  -> saved: exported_plots/model_comparison_summary.txt

=== All exported PNG files ===
  exported_plots\model_comparison_4in1.png
  exported_plots\model_comparison_inference_speed.png
  exported_plots\model_comparison_map50_95.png
  exported_plots\training_curves_side_by_side.png
  exported_plots\yolov8n_confusion_matrix.png
  exported_plots\yolov8n_training_results.png
  exported_plots\yolov8s_confusion_matrix.png
  exported_plots\yolov8s_training_results.png


In [ ]:
# 7. SAMPLE PREDICTIONS ON TEST SET 
test_img_dir = data_config['test']
if not os.path.isabs(test_img_dir):
    test_img_dir = os.path.join(os.path.dirname(DATA_PATH), test_img_dir)

pred_results = best_model.predict(
    source=test_img_dir,
    save=True,
    conf=0.25,
    name='test_predictions_yolov8s',
    seed=RANDOM_STATE,
    exist_ok=True,
)

pred_dir = str(pred_results[0].save_dir)
pred_images = sorted(glob.glob(os.path.join(pred_dir, '*.jpg')))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flatten(), pred_images):
    ax.imshow(Image.open(img_path))
    ax.axis('off')
plt.suptitle('Sample Predictions on Test Set (YOLOv8s)', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'sample_predictions_yolov8s.png')
plt.show()



image 1/212 C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL\test\images\IMG_0013_JPG.rf.e5e9aa325203a0ecc7a1886cfef5d21c.jpg: 640x640 2 Mobils, 1 Motor, 1 Plat, 52.4ms
image 2/212 C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL\test\images\IMG_0036_JPG.rf.03314a8c678a032106abc8a72da6a472.jpg: 640x640 1 Bus Besar, 37.7ms
image 3/212 C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL\test\images\IMG_0094_JPG.rf.7be21be9100d167d6c7fc67a62c507f9.jpg: 640x640 1 Angkot, 35.9ms
image 4/212 C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL\test\images\IMG_0100_JPG.rf.06e86a29cb21bb3cd9aebf1e5dba2966.jpg: 640x640 1 Bus Sedang, 6 Mobils, 4 Motors, 2 Plats, 34.4ms
image 5/212 C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL\test\images\IMG_0110_JPG.rf.d0a31bcb6e692dd34e735453a7709b80.jpg: 640x640 2 Mobils, 1 Motor, 1 Truk Sedang, 34.6ms
image 6/212 C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL\test\images\IMG_1807_JP

<Figure size 1500x1000 with 6 Axes>

In [16]:
import gradio as gr

# Use the best model selected from the comparison section if available.
# If the comparison cell has not been run, fallback to the main YOLOv8s model.
DEPLOY_MODEL_PATH = globals().get(
    'DEPLOY_MODEL_PATH',
    globals().get('BEST_MODEL_PATH', r'runs/detect/vehicle_detection/weights/best.pt')
)

model = YOLO(DEPLOY_MODEL_PATH)
print(f"Loaded deployment model: {DEPLOY_MODEL_PATH}")

def detect(img):
    results = model.predict(source=img, conf=0.25, seed=RANDOM_STATE)
    return Image.fromarray(results[0].plot())

demo = gr.Interface(
    fn=detect,
    inputs=gr.Image(type="pil", label="Upload Gambar"),
    outputs=gr.Image(type="pil", label="Hasil Deteksi"),
    title="Vehicle & License Plate Detection",
    description="Upload foto kendaraan untuk mendeteksi jenis kendaraan dan platnya"
)

demo.launch()


Loaded deployment model: C:\Samuel\Kuliah\Semester 4\Deep Learning (DL)\Tugas\DL_FINAL\runs\detect\vehicle_detection\weights\best.pt
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# Final Conclusion: Is the Model Ready for Real-World Use?

## Summary of Results

| Metric | YOLOv8s | YOLOv8n |
|---|---|---|
| Parameters | 11.14M | 3.01M |
| mAP50 | 0.831 | 0.751 |
| mAP50-95 | 0.577 | 0.523 |
| Precision | 0.827 | 0.709 |
| Recall | 0.756 | 0.696 |
| Inference (ms/img) | 17.73 | 8.21 |

YOLOv8s wins on every accuracy metric, while YOLOv8n is 2.16x faster and 3.7x lighter. We select **YOLOv8s as the final model** because accuracy is the priority for this project, while noting that YOLOv8n is a strong candidate when fast inference or lightweight deployment is required.

## Readiness Assessment

This model is a **working proof of concept**, not a production-ready system. With a recall of 0.76, roughly 1 out of every 4 real vehicles is still missed.

**Suitable for now (non-critical use):**
- Traffic surveys and approximate vehicle counting
- Transportation pattern studies (e.g. motorcycle vs car dominance)
- Demos and educational tools

**Not yet suitable (critical use):**
- Automated traffic law enforcement (missed detections mean violators escape)
- Billing or retribution systems that require accurate counts
- Safety-critical systems

## Limitations and Future Work

- Minority classes (`Becak`, `Gerobag`, `Truk Besar`) have too few samples, leading to low recall.
- The model produces notable false positives and false negatives on `Motor`, suggesting a need for hard-negative mining.
- Future improvements: add more data for minority and small-object classes, tune the confidence threshold per use case, train longer, or use a larger model (YOLOv8m/l) if deployment hardware allows.